
# Stage 2 — Train MLP on FMCS Dataset (FMCE / FMCS Pipeline)

This notebook assumes you have **already run Stage 1** to build an FMCS dataset, i.e.:
- Extracted feature maps from each epoch checkpoint (or FMCE phase).
- Saved them as:
  - `fmcs_features_layer-1.npy`
  - `fmcs_labels_layer-1.npy`

Here we:
1. Load the FMCS feature matrix `X` and label vector `y`.
2. Split into train / validation sets.
3. Standardize the features (using train statistics).
4. Train a simple MLP classifier to predict the **epoch/phase label**.
5. Report training / validation accuracy and save the best MLP checkpoint.


In [ ]:

#@title ⚙️ Config & Imports

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class CFG:
    # 🔴 TODO: 修改成你实际的 FMCS 目录（Stage1 保存 FMCS 的路径）
    fmcs_dir = "/content/drive/MyDrive/ExplainableNLP/FMCE_dataset_25"
    feature_file = "fmcs_features_layer-1.npy"  # must match Stage1
    label_file   = "fmcs_labels_layer-1.npy"    # must match Stage1

    val_ratio = 0.2          # 80% 训练, 20% 验证
    batch_size = 512
    num_epochs = 50
    lr = 1e-3
    hidden_dims = [512, 256]  # MLP 隐藏层维度
    device = "cuda" if torch.cuda.is_available() else "cpu"
    seed = 42

print("Using device:", CFG.device)
print("FMCS directory:", CFG.fmcs_dir)


In [ ]:

#@title 🔒 Set Random Seed

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)
print("Seed set to", CFG.seed)


In [ ]:

#@title 1️⃣ Load FMCS Features & Labels

feature_path = os.path.join(CFG.fmcs_dir, CFG.feature_file)
label_path   = os.path.join(CFG.fmcs_dir, CFG.label_file)

print("Feature path:", feature_path)
print("Label path  :", label_path)
print("Feature exists?", os.path.exists(feature_path))
print("Label exists?  ", os.path.exists(label_path))

if not os.path.exists(feature_path) or not os.path.exists(label_path):
    raise FileNotFoundError(
        "FMCS data not found. Please check CFG.fmcs_dir / feature_file / label_file."
    )

X = np.load(feature_path).astype(np.float32)  # [N, D]
y = np.load(label_path).astype(np.int64)      # [N]

print("Loaded features:", X.shape)
print("Loaded labels  :", y.shape, "min =", y.min(), "max =", y.max())
unique_labels = np.unique(y)
print("Original unique labels:", unique_labels)

# 原论文中 label 一般是 epoch 或 phase ID：1..K
# 为了适配 CrossEntropyLoss，这里转换为 0..K-1
y_shift = y.min()
y = y - y_shift
num_classes = int(len(np.unique(y)))
print(f"Remapped labels from [{y_shift}..] to [0..{num_classes-1}]")
print("New unique labels:", np.unique(y))


In [ ]:

#@title 2️⃣ Train / Val Split & Standardization

N = X.shape[0]
indices = np.arange(N)
np.random.shuffle(indices)

split = int(N * (1.0 - CFG.val_ratio))
train_idx = indices[:split]
val_idx   = indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]

print("Train size:", X_train.shape[0])
print("Val size  :", X_val.shape[0])

# 以 train 为基准做标准化
mean = X_train.mean(axis=0, keepdims=True)
std  = X_train.std(axis=0, keepdims=True) + 1e-8

X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std

print("Standardization done.")


In [ ]:

#@title 3️⃣ Build Dataset & DataLoader

class FMCSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FMCSDataset(X_train, y_train)
val_dataset   = FMCSDataset(X_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG.batch_size, shuffle=False)

print("Dataloaders ready.")


In [ ]:

#@title 4️⃣ Define MLP Model

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

input_dim = X.shape[1]
model = MLP(input_dim=input_dim,
            hidden_dims=CFG.hidden_dims,
            num_classes=num_classes).to(CFG.device)

print(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)


In [ ]:

#@title 5️⃣ Train & Validate MLP

import time

def run_epoch(loader, train: bool):
    if train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(CFG.device)
        batch_y = batch_y.to(CFG.device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == batch_y).sum().item()
        total += batch_x.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc

best_val_acc = 0.0
best_state_dict = None

start_time = time.time()
for epoch in range(1, CFG.num_epochs + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    t1 = time.time()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"[Epoch {epoch:02d}] "
          f"Train loss={train_loss:.4f}, acc={train_acc:.4f} | "
          f"Val loss={val_loss:.4f}, acc={val_acc:.4f} | "
          f"time={t1-t0:.2f}s")

total_time = time.time() - start_time
print(f"\n⏱ Total training time: {total_time:.2f}s")
print(f"Best val acc = {best_val_acc:.4f}")


In [ ]:

#@title 6️⃣ Save Best MLP Checkpoint

ckpt_path = os.path.join(CFG.fmcs_dir, "mlp_fmcs_best.pt")
if best_state_dict is not None:
    torch.save({
        "model_state_dict": best_state_dict,
        "input_dim": input_dim,
        "hidden_dims": CFG.hidden_dims,
        "num_classes": num_classes,
        "mean": mean,
        "std": std,
        "label_shift": int(y_shift),  # 原始 label = predicted_label + label_shift
    }, ckpt_path)
    print("✅ Saved best MLP checkpoint to:", ckpt_path)
else:
    print("⚠ No best_state_dict recorded, not saving model.")
